# 03 — Stock selection, persistent watchlists, and probability calibration

This notebook adds two pieces to the Cobasket workflow:

1. **Initial selection:** identify groups of stocks whose historical prices contain a plausible cointegrating relation.
2. **Repeated monitoring:** save those groups in a watchlist and re-evaluate every ticker even after its current holding becomes zero.

It also demonstrates **walk-forward calibration**. This repeatedly fits the model using only data available at each historical date, then checks what happened afterward. The resulting probability is empirical and conditional on the chosen basket, horizon, data period, and model settings.

> This is a research and decision-support workflow, not personalized financial advice or a guarantee of future returns.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from cobasket.data import DataManager
from cobasket.evidence import (
    BasketWatchlist,
    calibrate_evidence,
    calibrated_recommendation_table,
    calibration_table,
    candidate_table,
    cointegration_evidence,
    evaluate_watchlist,
    fit_probability_calibration,
    recommend_calibrated_assets,
    select_candidate_baskets,
    walk_forward_evidence,
    watchlist_from_candidates,
)

pd.set_option("display.max_colwidth", 120)

## 1. Download an initial universe

The **universe** is the collection of stocks Cobasket is allowed to consider. It is not the same thing as your portfolio.

- The portfolio contains stocks you currently own.
- The universe/watchlist contains stocks the software continues to monitor.

This separation is important. If you sell all shares in one ticker, set its holding to zero but leave it in the watchlist. Cobasket can then identify a later re-entry opportunity.

For a quick example we use a small, related group plus `SPY` as a broad US-market proxy. A real screen can use a larger, deliberately chosen universe.

In [ ]:
tickers = ["AAPL", "MSFT", "GOOGL", "META", "AMZN", "SPY"]
manager = DataManager(cache_dir="../price_cache")
universe_prices = manager.prices(tickers, period="5y", min_coverage=0.95)
universe_prices.tail()

## 2. Generate candidate baskets

The selector performs two filters:

1. It clusters stocks whose returns move similarly after removing the common movement represented by `SPY`.
2. It applies the Johansen test to each cluster and keeps groups with sufficiently strong evidence of cointegration.

This is comparable to source selection in astronomy: it narrows a large catalogue to plausible targets, but it does not prove that every selected target will remain useful.

In [ ]:
candidates = select_candidate_baskets(
    universe_prices,
    market_ticker="SPY",
    distance_threshold=1.2,
    min_trace_ratio=1.0,
    max_basket_size=6,
)

candidate_table(candidates)

The `trace_ratio` is the Johansen trace statistic divided by its 95% critical value. Values above one pass that particular threshold. Larger values indicate stronger statistical evidence in this data set, but should not be interpreted as a probability of profit.

## 3. Create and save a persistent watchlist

The watchlist stores basket membership, not quantities owned. It can therefore survive portfolio changes.

In [ ]:
if not candidates:
    raise RuntimeError(
        "No basket passed the current thresholds. Try a longer period, a broader universe, "
        "or inspect the clustering threshold rather than forcing a result."
    )

watchlist = watchlist_from_candidates(
    candidates,
    name="Technology watchlist",
    top_n=3,
)
watchlist.save("technology_watchlist.json")
watchlist

## 4. Evaluate current holdings

A holdings mapping records the number of shares currently owned. A quantity of zero means **not held**, but it does not remove the ticker from the watchlist.

The recommendation wording changes with ownership:

- positive evidence + not held → `Buy` or `Strong buy`;
- positive evidence + held → `Add` or `Strong add`;
- neutral evidence + not held → `Watch`;
- neutral evidence + held → `Hold`;
- negative evidence + not held → `Wait` or `Avoid buying`;
- negative evidence + held → `Hold without adding` or `Consider reducing`.

The software is evaluating relative position inside the basket. It is not yet estimating intrinsic company value.

In [ ]:
holdings = {
    "AAPL": 2.0,
    "MSFT": 1.0,
    # A stock that was completely sold remains explicitly at zero.
    "GOOGL": 0.0,
}

current_prices = universe_prices.loc[:, list(watchlist.tickers)]
evaluation = evaluate_watchlist(
    current_prices,
    watchlist,
    holdings=holdings,
    window=60,
)

pd.DataFrame([
    {
        "ticker": item.ticker,
        "action": item.action,
        "score": item.score,
        "confidence": item.confidence,
        "currently_held": item.currently_held,
    }
    for item in evaluation.recommendations
])

To model a complete sale, change the quantity to zero and run the same evaluation again. The ticker is still processed because membership comes from `watchlist`, not `holdings`.

In [ ]:
holdings_after_sale = dict(holdings)
holdings_after_sale["AAPL"] = 0.0

reentry_check = evaluate_watchlist(
    current_prices,
    watchlist,
    holdings=holdings_after_sale,
    window=60,
)

pd.DataFrame([
    {
        "ticker": item.ticker,
        "action": item.action,
        "currently_held": item.currently_held,
    }
    for item in reentry_check.recommendations
])

## 5. Walk-forward calibration

A single current score is not a probability. To estimate one, Cobasket walks through history:

1. Take a training window ending at date $t$.
2. Fit the cointegration relation using only data at or before $t$.
3. Produce an evidence score at $t$.
4. Move forward by the chosen horizon.
5. Record whether each stock outperformed the equal-weight return of its basket.
6. Repeat at later dates.

This is analogous to injection–recovery or cross-validation: the future segment is withheld until after the prediction has been made.

Here, “outperform” is **relative**:

$$
r_{\mathrm{excess},i} = r_i - \frac{1}{N}\sum_j r_j.
$$

A stock can outperform its basket while still falling in absolute price, for example if it falls by 2% while the basket average falls by 5%.

In [ ]:
basket = watchlist.baskets[0]
basket_prices = universe_prices.loc[:, list(basket)]

records = walk_forward_evidence(
    basket_prices,
    train_window=252,
    z_window=60,
    horizon=20,
    step=5,
    min_trace_ratio=1.0,
)

records.head()

`horizon=20` means roughly twenty trading days, not twenty calendar days. `step=5` issues a historical prediction approximately once per trading week. Overlapping horizons create correlated outcomes, so the apparent sample count is not the same as the number of fully independent experiments. We will address this more carefully when validating the full recommendation system.

In [ ]:
calibration = fit_probability_calibration(
    records,
    horizon=20,
    credible_level=0.68,
)
calibration_table(calibration)

Each row is an evidence-score interval. The probability is estimated with a beta-binomial model:

- `sample_count` is the number of historical examples in the interval;
- `probability_mean` is the posterior mean probability of basket-relative outperformance;
- `probability_lower` and `probability_upper` form a credible interval;
- an empty bin returns the prior mean of 50% with a wide interval rather than making an unsupported claim.

The default prior is uniform, $\mathrm{Beta}(1,1)$.

## 6. Apply calibration to the current state

The current basket evidence can now be mapped to empirical probabilities. Non-neutral recommendations require both enough historical examples and, by default, a credible interval that excludes 50%. This deliberately suppresses strong recommendations when calibration is poorly constrained.

In [ ]:
current_result = cointegration_evidence(
    basket_prices,
    window=60,
)
calibrated = calibrate_evidence(
    current_result.asset_evidence,
    calibration,
)
calibrated_recommendations = recommend_calibrated_assets(
    calibrated,
    holdings=holdings_after_sale,
)
calibrated_recommendation_table(calibrated_recommendations)

## 7. Interpretation and limitations

The displayed probability means:

> Given historical cases generated by this exact walk-forward procedure, what fraction of similarly scored stocks outperformed the equal-weight basket over the next twenty trading observations?

It does **not** mean:

- probability that the stock price rises;
- probability that buying produces a profit after fees;
- probability that the company is fundamentally undervalued;
- probability that the historical cointegration relation survives.

The next modelling work should test calibration stability across time, baskets, horizons, and non-overlapping evaluation periods before these probabilities drive consequential decisions.